# EGX Feature Engineering

This notebook loads the raw EGX OHLCV CSV files, engineers a robust set of **scale-invariant** features, validates the result, and saves one Parquet file per ticker to `data/processed/`.

All price-derived features are normalized (ratios, returns, percentages) so the model learns market patterns — not absolute price levels.

## 1. Setup and Paths

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

TICKERS = ["COMI_CA", "HRHO_CA", "TMGH_CA", "SWDY_CA", "ORWE_CA"]
DATE_COLUMN = "Date"
RAW_DIR_CANDIDATES = [Path("../data/raw"), Path("data/raw")]
PROCESSED_DIR_CANDIDATES = [Path("../data/processed"), Path("data/processed")]

# Final feature columns saved to parquet — all scale-invariant
FEATURE_COLUMNS = [
    "Target",
    # --- Momentum & Trend ---
    "RSI_14",           # already 0-100 bounded, scale-free
    "MACD_Norm",        # MACD / Close
    "MACD_Signal_Norm", # Signal / Close
    "MACD_Hist_Norm",   # (MACD - Signal) / Close
    # --- Volatility ---
    "Bollinger_Width",    # (Upper - Lower) / Mid  — band width as % of price
    "Bollinger_Position", # (Close - Lower) / (Upper - Lower)  — where in band
    "ATR_Pct",            # ATR_14 / Close  — volatility as % of price
    # --- Price Memory (returns, not levels) ---
    "Return_Lag_1",   # 1-day return
    "Return_Lag_5",   # 5-day return
    "Return_Lag_10",  # 10-day return
    "Return_Lag_21",  # 21-day return
    # --- Rolling Structure ---
    "Close_MA5_Ratio",  # Close / 5-day MA  — how extended above short-term average
    "Close_MA21_Ratio", # Close / 21-day MA — how extended above medium-term average
    "Close_CV5",        # std / mean over 5 days  — short-term price dispersion
    "Close_CV21",       # std / mean over 21 days — medium-term price dispersion
    # --- Volume ---
    "Volume_Ratio",  # today's volume / 21-day average volume
    "Volume_Spike",  # binary: 1 if volume > 2.5x its 21-day average
    # --- Calendar / Seasonality ---
    "Day_Of_Week",  # 0=Monday … 4=Friday (EGX trades Sun-Thu; encodes session position)
    "Month",        # 1-12 — captures macro seasonality
    "is_Ramadan",   # binary: 1 if trading day falls inside Ramadan window
]


def resolve_existing_path(candidates: Iterable[Path]) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return next(iter(candidates))


RAW_DIR = resolve_existing_path(RAW_DIR_CANDIDATES)
PROCESSED_DIR = resolve_existing_path(PROCESSED_DIR_CANDIDATES)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory      : {RAW_DIR.resolve() if RAW_DIR.exists() else RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR.resolve() if PROCESSED_DIR.exists() else PROCESSED_DIR}")
print(f"Feature columns         : {len(FEATURE_COLUMNS) - 1} features + 1 target")

Raw data directory      : D:\my_projects\Egx-analyst\data\raw
Processed data directory: D:\my_projects\Egx-analyst\data\processed
Feature columns         : 20 features + 1 target


## 2. Feature Engineering Functions

All feature computation lives inside `add_technical_features()`. Raw intermediate values (e.g. MACD in EGP) are computed as needed but are **always normalized before being kept**. This single-function architecture translates directly into `src/features.py` with no restructuring.

In [ ]:
def calculate_rsi(series: pd.Series, window: int = 14) -> pd.Series:
    """Wilder-smoothed RSI. Returns values in [0, 100]. NaNs filled with 50 (neutral).
        rsi  -> relative strength index which measure the speed and change of price movements
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - (100 / (1 + rs))).fillna(50.0)


def calculate_atr(frame: pd.DataFrame, window: int = 14) -> pd.Series:
    """14-day Average True Range in raw price units (used internally; normalized before output).
    atr  -> average true range which measure market volatility by decomposing the entire range of an asset price for that period"""
    prev_close = frame["Close"].shift(1)
    tr = pd.concat(
        [
            frame["High"] - frame["Low"],
            (frame["High"] - prev_close).abs(),
            (frame["Low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    return tr.rolling(window=window, min_periods=window).mean()


def add_technical_features(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Compute all features from raw OHLCV data.

    Returns a DataFrame containing ONLY the columns in FEATURE_COLUMNS plus
    the raw OHLCV inputs. All price-denominated intermediate values are
    normalized before being included in the output.

    Architecture note: all feature logic lives here so this function can be
    imported identically by notebooks, src/features.py, and api/routers/predict.py.
    """
    f = frame.copy()

    # ------------------------------------------------------------------ #
    # TARGET
    # Did the closing price go UP the next trading day? 1 = yes, 0 = no.
    # shift(-1): tomorrow's close. The final row gets NaN → dropped later.
    # ------------------------------------------------------------------ #
    f["Target"] = (f["Close"].shift(-1) > f["Close"]).astype("Int64")

    # ------------------------------------------------------------------ #
    # RSI — already bounded [0, 100], scale-free
    # ------------------------------------------------------------------ #
    f["RSI_14"] = calculate_rsi(f["Close"], window=14)

    # ------------------------------------------------------------------ #
    # MACD ->  moving average convergence divergence that measures trend strength and direction by comparing two moving averages of price
    # MACD — compute raw, then normalize by Close to make scale-invariant
    # Raw MACD is in EGP; stocks at different price levels are not comparable.
    # Dividing by Close converts to a return-like fraction.
    # ------------------------------------------------------------------ #
    ema_12 = f["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = f["Close"].ewm(span=26, adjust=False).mean()
    macd_raw = ema_12 - ema_26
    signal_raw = macd_raw.ewm(span=9, adjust=False).mean()
    hist_raw = macd_raw - signal_raw
    close_safe = f["Close"].replace(0, np.nan)  # guard against division by zero
    f["MACD_Norm"] = macd_raw / close_safe
    f["MACD_Signal_Norm"] = signal_raw / close_safe
    f["MACD_Hist_Norm"] = hist_raw / close_safe  # crossover signal: positive = bullish

    # ------------------------------------------------------------------ #
    #   Bollinger Bands -> a volatility indicator that consists of a moving average (mid band) and two standard deviation bands (upper and lower).
    #   BOLLINGER BANDS — compute raw bands, then derive two normalized features
    #   Bollinger_Width    : band width as a fraction of mid price (volatility proxy)
    #   Bollinger_Position : where Close sits inside the band (0=lower, 1=upper, >1=breakout)
    # ------------------------------------------------------------------ #
    roll20 = f["Close"].rolling(window=20, min_periods=20)
    bb_mid = roll20.mean()
    bb_std = roll20.std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std
    f["Bollinger_Width"] = (bb_upper - bb_lower) / bb_mid.replace(0, np.nan)
    f["Bollinger_Position"] = (f["Close"] - bb_lower) / ((bb_upper - bb_lower) + 1e-9)

    # ------------------------------------------------------------------ #
    # ATR — compute raw, then normalize by Close
    # ATR in EGP is meaningless across stocks or across a stock's own price history.
    # ATR_Pct expresses daily volatility as a % of current price.
    # ------------------------------------------------------------------ #
    atr_raw = calculate_atr(f, window=14)
    f["ATR_Pct"] = atr_raw / close_safe

    # ------------------------------------------------------------------ #
    # RETURN LAGS — percentage returns, not price levels
    # pct_change(n) = (Close_today / Close_{today-n}) - 1
    # ------------------------------------------------------------------ #
    for lag in [1, 5, 10, 21]:
        f[f"Return_Lag_{lag}"] = f["Close"].pct_change(lag)

    # ------------------------------------------------------------------ #
    # ROLLING STRUCTURE — ratios and coefficients of variation
    #   Close_MAn_Ratio : how far price has extended above its n-day average (>1 = above)
    #   Close_CVn       : std / mean over n days — relative price dispersion
    # ------------------------------------------------------------------ #
    for window in [5, 21]:
        roll = f["Close"].rolling(window=window, min_periods=window)
        roll_mean = roll.mean()
        f[f"Close_MA{window}_Ratio"] = f["Close"] / roll_mean.replace(0, np.nan)
        f[f"Close_CV{window}"] = roll.std() / roll_mean.replace(0, np.nan)

    # ------------------------------------------------------------------ #
    # VOLUME FEATURES
    #   Volume_Ratio : today's volume relative to its 21-day average
    #   Volume_Spike : binary flag when Volume_Ratio > 2.5 (institutional activity)
    # Raw Volume_MA_21 is kept as an intermediate only.
    # ------------------------------------------------------------------ #
    vol_ma21 = f["Volume"].rolling(window=21, min_periods=21).mean()
    f["Volume_Ratio"] = f["Volume"] / vol_ma21.replace(0, np.nan)
    f["Volume_Spike"] = (f["Volume"] > 2.5 * vol_ma21).astype("Int64")

    # ------------------------------------------------------------------ #
    # CALENDAR FEATURES — seasonality signals
    # Day_Of_Week: EGX trades Sun(6)-Thu(3) on the Gregorian calendar;
    #   pandas dayofweek gives 0=Mon…6=Sun, so EGX days land on 0,1,2,3,6.
    # Month: captures macro seasonality (post-Ramadan, summer, fiscal year-end).
    # ------------------------------------------------------------------ #
    f["Day_Of_Week"] = f.index.dayofweek.astype("Int64")
    f["Month"] = f.index.month.astype("Int64")

    return f

## 3. Ramadan and Holiday Flags

These are EGX-specific seasonality signals. Ramadan creates measurable liquidity changes (shortened sessions, reduced institutional activity). The `holidays` library is used when available; if not, the manually curated `CUSTOM_HOLIDAY_MAP` serves as a fallback.

**Note on `is_Egypt_Holiday`:** Since zero-volume rows were already removed during raw data download, official exchange closure days do not appear in the dataset. After testing, this flag was found to be near-zero across all tickers and is excluded from model features to avoid noise.

In [3]:
RAMADAN_RANGES = {
    2019: ("2019-05-06", "2019-06-04"),
    2020: ("2020-04-24", "2020-05-23"),
    2021: ("2021-04-13", "2021-05-12"),
    2022: ("2022-04-02", "2022-05-01"),
    2023: ("2023-03-23", "2023-04-21"),
    2024: ("2024-03-11", "2024-04-09"),
    2025: ("2025-03-01", "2025-03-29"),
    2026: ("2026-02-18", "2026-03-19"),
}

CUSTOM_HOLIDAY_MAP = {
    2019: {
        "fixed": ["2019-01-07", "2019-01-25", "2019-04-25", "2019-05-01", "2019-06-30", "2019-07-23", "2019-10-06"],
        "eid_al_fitr": ["2019-06-05", "2019-06-06", "2019-06-07"],
        "eid_al_adha": ["2019-08-12", "2019-08-13", "2019-08-14", "2019-08-15"],
    },
    2020: {
        "fixed": ["2020-01-07", "2020-01-25", "2020-04-25", "2020-05-01", "2020-06-30", "2020-07-23", "2020-10-06"],
        "eid_al_fitr": ["2020-05-24", "2020-05-25", "2020-05-26", "2020-05-27"],
        "eid_al_adha": ["2020-07-31", "2020-08-01", "2020-08-02", "2020-08-03"],
    },
    2021: {
        "fixed": ["2021-01-07", "2021-01-25", "2021-04-25", "2021-05-01", "2021-06-30", "2021-07-23", "2021-10-06"],
        "eid_al_fitr": ["2021-05-13", "2021-05-14", "2021-05-15"],
        "eid_al_adha": ["2021-07-20", "2021-07-21", "2021-07-22", "2021-07-23"],
    },
    2022: {
        "fixed": ["2022-01-07", "2022-01-25", "2022-04-25", "2022-05-01", "2022-06-30", "2022-07-23", "2022-10-06"],
        "eid_al_fitr": ["2022-05-02", "2022-05-03", "2022-05-04"],
        "eid_al_adha": ["2022-07-09", "2022-07-10", "2022-07-11", "2022-07-12"],
    },
    2023: {
        "fixed": ["2023-01-07", "2023-01-25", "2023-04-25", "2023-05-01", "2023-06-30", "2023-07-23", "2023-10-06"],
        "eid_al_fitr": ["2023-04-21", "2023-04-22", "2023-04-23"],
        "eid_al_adha": ["2023-06-27", "2023-06-28", "2023-06-29", "2023-06-30"],
    },
    2024: {
        "fixed": ["2024-01-07", "2024-01-25", "2024-04-25", "2024-05-01", "2024-06-30", "2024-07-23", "2024-10-06"],
        "eid_al_fitr": ["2024-04-10", "2024-04-11", "2024-04-12"],
        "eid_al_adha": ["2024-06-17", "2024-06-18", "2024-06-19", "2024-06-20"],
    },
    2025: {
        "fixed": ["2025-01-07", "2025-01-25", "2025-04-25", "2025-05-01", "2025-06-30", "2025-07-23", "2025-10-06"],
        "eid_al_fitr": ["2025-03-30", "2025-03-31", "2025-04-01"],
        "eid_al_adha": ["2025-06-06", "2025-06-07", "2025-06-08", "2025-06-09"],
    },
    2026: {
        "fixed": ["2026-01-07", "2026-01-25", "2026-04-25", "2026-05-01", "2026-06-30", "2026-07-23", "2026-10-06"],
        "eid_al_fitr": ["2026-03-20", "2026-03-21", "2026-03-22"],
        "eid_al_adha": ["2026-05-26", "2026-05-27", "2026-05-28", "2026-05-29"],
    },
}

try:
    import holidays  # type: ignore
except ImportError:
    holidays = None


def build_custom_holiday_lookup() -> set:
    holiday_dates: set = set()
    for year_data in CUSTOM_HOLIDAY_MAP.values():
        for date_group in year_data.values():
            holiday_dates.update(pd.to_datetime(date_group))
    return holiday_dates


def build_is_ramadan_flag(index: pd.DatetimeIndex) -> pd.Series:
    mask = pd.Series(False, index=index)
    for start_date, end_date in RAMADAN_RANGES.values():
        year_mask = (index >= pd.Timestamp(start_date)) & (index <= pd.Timestamp(end_date))
        mask = mask | year_mask
    return mask.astype("Int64")


def build_is_egypt_holiday_flag(index: pd.DatetimeIndex) -> pd.Series:
    if holidays is not None:
        try:
            egypt_holidays = holidays.Egypt()
        except AttributeError:
            egypt_holidays = holidays.EG()
        return pd.Series(
            index.to_series().dt.date.map(lambda d: int(pd.Timestamp(d) in egypt_holidays)),
            index=index,
        ).astype("Int64")

    holiday_lookup = build_custom_holiday_lookup()
    return pd.Series(
        index.to_series().map(lambda ts: int(pd.Timestamp(ts.normalize()) in holiday_lookup)),
        index=index,
    ).astype("Int64")

## 4. Batch Processing, Validation, and Export

In [4]:
def engineer_features_for_ticker(ticker: str) -> pd.DataFrame:
    """
    Full pipeline for one ticker:
      load CSV → validate schema → engineer features → add calendar flags
      → drop NaNs → select final columns → save parquet
    """
    # --- Load ---
    source_path = RAW_DIR / f"{ticker}.csv"
    if not source_path.exists():
        raise FileNotFoundError(f"Missing raw file: {source_path}")

    frame = pd.read_csv(source_path, parse_dates=[DATE_COLUMN])
    frame = frame.rename(columns={col: col.strip() for col in frame.columns})
    frame = frame.sort_values(DATE_COLUMN).set_index(DATE_COLUMN)

    # --- Schema validation ---
    expected = {"Close", "High", "Low", "Open", "Volume"}
    missing = expected.difference(frame.columns)
    if missing:
        raise ValueError(f"{ticker}: missing required columns {sorted(missing)}")

    frame = frame.loc[:, ["Close", "High", "Low", "Open", "Volume"]].copy()
    frame = frame.astype(
        {"Close": "float64", "High": "float64", "Low": "float64", "Open": "float64", "Volume": "float64"}
    )

    # --- Feature engineering (all logic lives in add_technical_features) ---
    frame = add_technical_features(frame)

    # --- Calendar flags ---
    frame["is_Ramadan"] = build_is_ramadan_flag(frame.index)

    # Diagnostic: check if holiday flag is useful (expected ~0 since zero-volume
    # rows were already filtered out during raw data download)
    holiday_flag = build_is_egypt_holiday_flag(frame.index)
    print(f"  [diag] is_Egypt_Holiday sum = {int(holiday_flag.sum())} "
          f"({'kept' if holiday_flag.sum() > 0 else 'excluded — all zeros after volume filter'})")
    # Not added to FEATURE_COLUMNS because it carries no signal after zero-volume rows are removed.

    # --- Drop NaN rows (from rolling windows + shift(-1) on target) ---
    frame = frame.dropna(subset=FEATURE_COLUMNS).copy()

    # --- Select only final feature columns ---
    frame = frame.loc[:, FEATURE_COLUMNS].copy()

    # --- Validation summary ---
    print(f"\n{'='*55}")
    print(f"  {ticker} — feature engineering summary")
    print(f"{'='*55}")
    print(f"  Rows after cleaning : {len(frame)}")
    print(f"  Columns             : {frame.shape[1]}  ({frame.shape[1]-1} features + 1 target)")
    print(f"  Date range          : {frame.index[0].date()} → {frame.index[-1].date()}")
    print(f"  Target balance      : UP={int(frame['Target'].sum())} "
          f"({frame['Target'].mean()*100:.1f}%)  "
          f"DOWN={int((frame['Target']==0).sum())} "
          f"({(1-frame['Target'].mean())*100:.1f}%)")
    null_counts = frame.isna().sum()
    total_nulls = null_counts.sum()
    print(f"  Total null values   : {total_nulls} {'✓ clean' if total_nulls == 0 else '✗ FIX THIS'}")
    if total_nulls > 0:
        print(null_counts[null_counts > 0].to_string())
    print(f"  Feature ranges (min / max):")
    desc = frame[FEATURE_COLUMNS[1:]].describe().loc[["min", "max"]]
    print(desc.to_string())

    # --- Save ---
    output_path = PROCESSED_DIR / f"{ticker}_features.parquet"
    frame.to_parquet(output_path, index=True)
    print(f"\n  Saved → {output_path}")

    return frame


engineered_datasets: Dict[str, pd.DataFrame] = {}
for ticker in TICKERS:
    engineered_datasets[ticker] = engineer_features_for_ticker(ticker)

print("\n\n✓ Feature engineering complete for all tickers.")

  [diag] is_Egypt_Holiday sum = 12 (kept)

  COMI_CA — feature engineering summary
  Rows after cleaning : 1736
  Columns             : 21  (20 features + 1 target)
  Date range          : 2019-02-04 → 2026-06-29
  Target balance      : UP=870 (50.1%)  DOWN=866 (49.9%)
  Total null values   : 0 ✓ clean
  Feature ranges (min / max):
        RSI_14  MACD_Norm  MACD_Signal_Norm  MACD_Hist_Norm  Bollinger_Width  Bollinger_Position   ATR_Pct  Return_Lag_1  Return_Lag_5  Return_Lag_10  Return_Lag_21  Close_MA5_Ratio  Close_MA21_Ratio  Close_CV5  Close_CV21  Volume_Ratio  Volume_Spike  Day_Of_Week  Month  is_Ramadan
min  14.691664  -0.113109         -0.095304       -0.043213         0.017905           -0.455056  0.009420     -0.100840     -0.235264      -0.293765      -0.332394         0.882371          0.741529   0.000867    0.005078      0.046970           0.0          0.0    1.0         0.0
max  87.293461   0.075523          0.071700        0.025351         0.558681            1.390038  0.

## 5. Cross-Ticker Sanity Check

Verify that all 5 tickers have consistent schemas, no nulls, and similar feature value ranges — important because we train one model per ticker, and all must share the same feature contract.

In [5]:
print("Cross-ticker schema check")
print("-" * 50)

first_cols = None
for ticker, df in engineered_datasets.items():
    cols = list(df.columns)
    nulls = int(df.isna().sum().sum())
    target_bal = f"{df['Target'].mean()*100:.1f}% UP"
    schema_ok = (cols == FEATURE_COLUMNS)
    print(f"  {ticker:<10} rows={len(df)}  nulls={nulls}  target={target_bal}  schema={'✓' if schema_ok else '✗ MISMATCH'}")
    if first_cols is None:
        first_cols = cols
    elif cols != first_cols:
        print(f"    !! Column mismatch vs first ticker")

print("\nFeature value ranges across all tickers:")
combined = pd.concat(engineered_datasets.values())
display(combined[FEATURE_COLUMNS[1:]].describe().loc[["mean", "std", "min", "max"]].round(4))

Cross-ticker schema check
--------------------------------------------------
  COMI_CA    rows=1736  nulls=0  target=50.1% UP  schema=✓
  HRHO_CA    rows=1736  nulls=0  target=47.1% UP  schema=✓
  TMGH_CA    rows=1740  nulls=0  target=47.0% UP  schema=✓
  SWDY_CA    rows=1738  nulls=0  target=46.3% UP  schema=✓
  ORWE_CA    rows=1739  nulls=0  target=47.3% UP  schema=✓

Feature value ranges across all tickers:


,RSI_14,MACD_Norm,MACD_Signal_Norm,MACD_Hist_Norm,Bollinger_Width,Bollinger_Position,ATR_Pct,Return_Lag_1,Return_Lag_5,Return_Lag_10,Return_Lag_21,Close_MA5_Ratio,Close_MA21_Ratio,Close_CV5,Close_CV21,Volume_Ratio,Volume_Spike,Day_Of_Week,Month,is_Ramadan
mean,52.3353,0.0047,0.0048,-0.0000,0.1543,0.5329,0.0326,0.0012,0.0066,0.0134,0.0286,1.0021,1.0110,0.0190,0.0396,1.0214,0.0565,2.3544,6.4407,0.0903
std,12.7240,0.0282,0.0263,0.0087,0.1158,0.3277,0.0148,0.0239,0.0587,0.0852,0.1286,0.0278,0.0672,0.0161,0.0296,0.8817,0.2309,2.0528,3.4718,0.2867
min,8.2709,-0.2789,-0.2320,-0.0877,0.0179,-0.4551,0.0094,-0.1998,-0.3436,-0.4529,-0.5689,0.8045,0.5467,0.0009,0.0051,0.0157,0.0,0.0,1.0,0.0
max,92.4431,0.1498,0.1377,0.0548,1.1214,1.4490,0.1318,0.2000,0.6396,0.9260,1.4788,1.2594,1.6759,0.1834,0.2787,18.7859,1.0,6.0,12.0,1.0


## 6. Feature Reference

### Design principle: scale-invariance
Every feature is expressed as a ratio, percentage, or bounded index — never in raw EGP. This allows a single model to learn market patterns that hold regardless of whether a stock trades at EGP 5 or EGP 500, and regardless of how much prices have drifted over the 2019–2026 training window.

- MA -> simple moving average
- EMA -> exponential moving average
- EWM -> exponentially weighted moving average

- RSI -> relative strength index
- MACD -> moving average convergence divergence
- ATR -> average true range , when RIS > 70 stock heads to overbought territory, when RSI < 30 stock heads to panic selling territory
- CV -> coefficient of variation (std / mean)
- Bollinger Bands -> a volatility indicator that consists of a moving average (mid band) and two standard deviation bands (upper and lower).
- volume spike -> a sudden increase in trading volume, often indicating increased interest or activity in a stock.

### Feature groups

| Feature | Formula | Range | What it captures |
|---|---|---|---|
| `RSI_14` | Wilder EWM on 14-day gains/losses | 0–100 | Overbought (>70) / oversold (<30) momentum |
| `MACD_Norm` | (EMA12 − EMA26) / Close | ±small | Trend momentum direction |
| `MACD_Signal_Norm` | 9-day EMA of MACD / Close | ±small | Smoothed trend baseline |
| `MACD_Hist_Norm` | (MACD − Signal) / Close | ±small | Trend acceleration; positive = bullish crossover |
| `Bollinger_Width` | (Upper − Lower) / Mid | 0–∞ | Volatility regime; high = wide bands = high uncertainty |
| `Bollinger_Position` | (Close − Lower) / (Upper − Lower) | 0–1+ | Position within band; >1 = upper breakout |
| `ATR_Pct` | ATR14 / Close | 0–∞ | Daily volatility as % of price |
| `Return_Lag_1` | pct_change(1) | ±∞ | Yesterday's return (momentum signal) |
| `Return_Lag_5` | pct_change(5) | ±∞ | Last week's return |
| `Return_Lag_10` | pct_change(10) | ±∞ | Last 2-week return |
| `Return_Lag_21` | pct_change(21) | ±∞ | Last month's return |
| `Close_MA5_Ratio` | Close / MA5 | ~0.8–1.2 | Extension above short-term average |
| `Close_MA21_Ratio` | Close / MA21 | ~0.7–1.3 | Extension above medium-term average |
| `Close_CV5` | STD5 / MA5 | 0–∞ | Short-term price dispersion |
| `Close_CV21` | STD21 / MA21 | 0–∞ | Medium-term price dispersion |
| `Volume_Ratio` | Volume / VolMA21 | 0–∞ | Relative trading activity |
| `Volume_Spike` | 1 if Volume_Ratio > 2.5 | 0 or 1 | Institutional activity flag |
| `Day_Of_Week` | pandas dayofweek | 0–6 | Session position in trading week |
| `Month` | 1–12 | 1–12 | Macro seasonality |
| `is_Ramadan` | lookup in RAMADAN_RANGES | 0 or 1 | EGX-specific liquidity regime |